# Pipeline completo - Base 03

Responsavel: Matheus Makita.

Executar toda a analise e os quatro modelos neste notebook.


In [ ]:
from pathlib import Path
from time import perf_counter
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss

BASE_ID = "base_03"
RESPONSAVEL = "Matheus Makita"
DATA_PATH = Path("../data/raw/base_03/PRSA_Data_Aotizhongxin_20130301-20170228.csv")
TARGET = "PM2.5"
DATETIME_COL = "data_hora"
FREQ = "h"
HORIZON = 1
SEASONAL_PERIOD = 24
RANDOM_STATE = 42
TRAIN_FRACTION = .80
VALIDATION_FRACTION = .20
WARMUP = 168

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
DATA_PATH = PROJECT_ROOT / "data/raw/base_03/PRSA_Data_Aotizhongxin_20130301-20170228.csv"
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
PREDICTIONS_DIR = PROJECT_ROOT / "results/predictions"
METRICS_DIR = PROJECT_ROOT / "results/metrics"
RESIDUALS_DIR = PROJECT_ROOT / "results/residuals"
IMPORTANCE_DIR = PROJECT_ROOT / "results/feature_importance"
FIGURES_DIR = PROJECT_ROOT / "results/figures/base_03"
for pasta in [PROCESSED_DIR, PREDICTIONS_DIR, METRICS_DIR, RESIDUALS_DIR, IMPORTANCE_DIR, FIGURES_DIR]:
    pasta.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_STATE)
plt.rcParams.update({"figure.figsize": (13, 4), "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False})

## 1. Documentacao e auditoria


In [ ]:
df = pd.read_csv(DATA_PATH)
df[DATETIME_COL] = pd.to_datetime(df[["year", "month", "day", "hour"]], errors="raise")
assert not df[DATETIME_COL].duplicated().any()
dados = df.set_index(DATETIME_COL).sort_index().asfreq(FREQ)
assert dados.index.is_monotonic_increasing and dados.index.is_unique
assert (dados.index.to_series().diff().dropna() == pd.Timedelta(hours=1)).all()

auditoria = pd.DataFrame({
    "tipo": dados.dtypes.astype(str),
    "ausentes": dados.isna().sum(),
    "ausentes_%": dados.isna().mean().mul(100),
    "distintos": dados.nunique(),
})
display(pd.Series({"base": BASE_ID, "linhas": len(dados), "colunas": dados.shape[1],
                   "início": dados.index.min(), "fim": dados.index.max(),
                   "frequência": FREQ, "alvo": TARGET}, name="documentação"))
display(auditoria.round(3))

## 2. Limpeza, EDA e STL


In [ ]:
y = dados[TARGET]
corte_treino = int(len(dados) * TRAIN_FRACTION)
corte_validacao = int(corte_treino * (1 - VALIDATION_FRACTION))
idx_ajuste = dados.index[WARMUP:corte_validacao]
idx_validacao = dados.index[corte_validacao:corte_treino]
idx_treino = dados.index[WARMUP:corte_treino]
idx_teste = dados.index[corte_treino:]

display(pd.DataFrame({
    "início": [dados.index[0], idx_ajuste.min(), idx_validacao.min(), idx_teste.min()],
    "fim": [dados.index[-1], idx_ajuste.max(), idx_validacao.max(), idx_teste.max()],
    "observações": [len(dados), len(idx_ajuste), len(idx_validacao), len(idx_teste)],
}, index=["base", "ajuste interno", "validação interna", "teste final"]))

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(y, linewidth=.4)
ax.axvline(idx_teste.min(), color="red", linestyle="--", label="20% teste")
ax.set(title="PM2.5 horário", ylabel="µg/m³")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "serie_treino_teste.png", dpi=160)
plt.show()

serie_diagnostico = y.iloc[:corte_treino].interpolate(method="time", limit_direction="both")
stl = STL(serie_diagnostico, period=SEASONAL_PERIOD, robust=True).fit()
forca_sazonal = max(0, 1 - np.var(stl.resid) / np.var(stl.seasonal + stl.resid))
display(pd.Series({"período sazonal": SEASONAL_PERIOD, "força da sazonalidade": forca_sazonal}, name="STL"))

fig, axs = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
for ax, valores, titulo in zip(axs, [serie_diagnostico, stl.trend, stl.seasonal, stl.resid],
                               ["Observada", "Tendência", "Sazonalidade", "Resíduo"]):
    ax.plot(valores, linewidth=.6)
    ax.set_ylabel(titulo)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "stl.png", dpi=160)
plt.show()

In [ ]:
def diagnostico_estacionariedade(serie, nome):
    serie = pd.Series(serie).dropna()
    adf = adfuller(serie, autolag="AIC")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        resultado_kpss = kpss(serie, regression="c", nlags="auto")
    return {"série": nome, "ADF p-valor": adf[1], "KPSS p-valor": resultado_kpss[1],
            "ADF estacionária": adf[1] < .05, "KPSS estacionária": resultado_kpss[1] > .05}

serie_diff = serie_diagnostico.diff().dropna()
serie_diff_sazonal = serie_diagnostico.diff(SEASONAL_PERIOD).dropna()
serie_diff_dupla = serie_diagnostico.diff(SEASONAL_PERIOD).diff().dropna()
display(pd.DataFrame([
    diagnostico_estacionariedade(serie_diagnostico, "original"),
    diagnostico_estacionariedade(serie_diff, "d=1"),
    diagnostico_estacionariedade(serie_diff_sazonal, "D=1, m=24"),
    diagnostico_estacionariedade(serie_diff_dupla, "d=1, D=1, m=24"),
]).set_index("série").round(4))

fig, axs = plt.subplots(2, 2, figsize=(15, 9))
plot_acf(serie_diagnostico, lags=72, ax=axs[0, 0], title="ACF — original")
plot_pacf(serie_diagnostico, lags=72, method="ywm", ax=axs[0, 1], title="PACF — original")
plot_acf(serie_diff_dupla, lags=72, ax=axs[1, 0], title="ACF — d=1 e D=1")
plot_pacf(serie_diff_dupla, lags=72, method="ywm", ax=axs[1, 1], title="PACF — d=1 e D=1")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "acf_pacf.png", dpi=160)
plt.show()

## 3. Feature engineering sem vazamento


In [ ]:
EXTERNAS = ["PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
DIRECOES = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE", "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"]

def criar_features(base):
    X = pd.DataFrame(index=base.index)
    for lag in [1, 2, 3, 6, 12, 24, 48, 168]:
        X[f"PM2.5_lag_{lag}"] = base[TARGET].shift(lag)
    passado = base[TARGET].shift(HORIZON)
    for janela in [6, 24, 168]:
        X[f"PM2.5_media_{janela}"] = passado.rolling(janela, min_periods=1).mean()
        X[f"PM2.5_std_{janela}"] = passado.rolling(janela, min_periods=2).std()
        X[f"PM2.5_validos_{janela}"] = passado.rolling(janela, min_periods=1).count()
    for coluna in EXTERNAS:
        for lag in [1, 24]:
            X[f"{coluna}_lag_{lag}"] = base[coluna].shift(lag)
    angulo = base["wd"].map({v: i*2*np.pi/16 for i, v in enumerate(DIRECOES)}).shift(1)
    X["vento_sin_lag1"], X["vento_cos_lag1"] = np.sin(angulo), np.cos(angulo)
    for nome, valores, periodo in [("hora", base.index.hour, 24), ("semana", base.index.dayofweek, 7),
                                   ("ano", base.index.dayofyear-1, 365.25)]:
        X[f"{nome}_sin"] = np.sin(2*np.pi*valores/periodo)
        X[f"{nome}_cos"] = np.cos(2*np.pi*valores/periodo)
    return X

X = criar_features(dados)
EXOG_SARIMAX = ["TEMP_lag_1", "PRES_lag_1", "WSPM_lag_1", "CO_lag_1", "PM10_lag_1",
                "hora_sin", "hora_cos", "ano_sin", "ano_cos"]

t = corte_treino
alterada = dados.copy()
alterada.loc[alterada.index[t]:, [TARGET] + EXTERNAS] = 99999
alterada.loc[alterada.index[t]:, "wd"] = "S"
pd.testing.assert_frame_equal(X.iloc[:t+1], criar_features(alterada).iloc[:t+1])
assert idx_treino.max() < idx_teste.min()

exog_vif = X.loc[idx_treino, EXOG_SARIMAX].fillna(X.loc[idx_treino, EXOG_SARIMAX].median())
exog_vif = StandardScaler().fit_transform(exog_vif)
display(pd.DataFrame({"variável": EXOG_SARIMAX,
                      "VIF": [variance_inflation_factor(exog_vif, i) for i in range(len(EXOG_SARIMAX))]}).round(3))
display(pd.Series({"features": X.shape[1], "causalidade verificada": True,
                   "alvos observados no teste": y.loc[idx_teste].notna().sum()}))

## 4. Walk-forward e otimizacao

Previsao de uma hora com origem movel: o instante `t` usa somente informacoes disponiveis ate `t-1`. O valor real de `t` entra no historico somente depois da previsao.


In [ ]:
def mae_observado(indices, previsao):
    real = y.loc[indices]
    previsao = np.asarray(previsao).reshape(-1)
    mascara = real.notna() & np.isfinite(previsao)
    return mean_absolute_error(real[mascara], previsao[mascara])

def pipeline_tabular(modelo):
    return TransformedTargetRegressor(
        regressor=Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
            ("modelo", modelo),
        ]),
        transformer=StandardScaler(),
    )

grades = {
    "Random Forest": [
        {"n_estimators": 150, "max_depth": 12, "min_samples_split": 4, "min_samples_leaf": 2, "max_features": .8},
        {"n_estimators": 250, "max_depth": 20, "min_samples_split": 2, "min_samples_leaf": 1, "max_features": 1.0},
    ],
    "SVR": [
        {"C": 1., "epsilon": .1, "gamma": "scale"},
        {"C": 10., "epsilon": .1, "gamma": "scale"},
    ],
}

def criar_modelo(nome, parametros):
    if nome == "Random Forest":
        estimador = RandomForestRegressor(**parametros, random_state=RANDOM_STATE, n_jobs=-1)
    else:
        estimador = SVR(**parametros, kernel="rbf", cache_size=1000)
    return pipeline_tabular(estimador)

def prever_ml_walk_forward(modelo, indices):
    # Cada linha contém apenas lags/rolling encerrados em t-1; o lote é
    # computacionalmente equivalente a avançar a origem uma hora por vez.
    indices = indices.sort_values()
    return np.asarray(modelo.predict(X.loc[indices])).reshape(-1)

previsoes = pd.DataFrame({"valor_real": y.loc[idx_teste]}, index=idx_teste)
modelos, melhores_parametros, tempos, busca = {}, {}, {}, []

## 5. SARIMAX


In [ ]:
def ajustar_sarimax(indices, parametros):
    preparador = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    exog = preparador.fit_transform(X.loc[indices, EXOG_SARIMAX])
    modelo = SARIMAX(y.loc[indices] / 100, exog=exog, order=parametros["order"],
                     seasonal_order=parametros["seasonal_order"], trend="c",
                     enforce_stationarity=False, enforce_invertibility=False)
    ajuste = modelo.fit(disp=False, maxiter=15, low_memory=True)
    return ajuste, preparador

def prever_sarimax_walk_forward(ajuste, preparador, treino, futuro):
    # fittedvalues do filtro de Kalman são previsões 1-step antes da atualização por y_t.
    todos = treino.append(futuro)
    exog = preparador.transform(X.loc[todos, EXOG_SARIMAX])
    modelo = SARIMAX(y.loc[todos] / 100, exog=exog, order=ajuste.model.order,
                     seasonal_order=ajuste.model.seasonal_order, trend="c",
                     enforce_stationarity=False, enforce_invertibility=False)
    filtrado = modelo.filter(ajuste.params, low_memory=True)
    return np.asarray(filtrado.fittedvalues[-len(futuro):]) * 100

candidatos_sarimax = [
    {"order": (1, 0, 0), "seasonal_order": (0, 0, 0, SEASONAL_PERIOD)},
    {"order": (1, 0, 1), "seasonal_order": (0, 0, 0, SEASONAL_PERIOD)},
    {"order": (2, 0, 0), "seasonal_order": (0, 0, 0, SEASONAL_PERIOD)},
    {"order": (2, 0, 1), "seasonal_order": (0, 0, 0, SEASONAL_PERIOD)},
    {"order": (2, 0, 2), "seasonal_order": (0, 0, 0, SEASONAL_PERIOD)},
]
inicio = perf_counter()
scores = []
idx_ajuste_sarimax = idx_ajuste[-8760:]
for parametros in candidatos_sarimax:
    with warnings.catch_warnings(record=True) as alertas:
        warnings.simplefilter("always")
        ajuste, preparador = ajustar_sarimax(idx_ajuste_sarimax, parametros)
    pred = prever_sarimax_walk_forward(ajuste, preparador, idx_ajuste_sarimax, idx_validacao)
    score = mae_observado(idx_validacao, pred)
    convergiu = ajuste.mle_retvals.get("converged", False)
    busca.append({"modelo": "SARIMAX", "parâmetros": parametros, "MAE_validação": score,
                  "AIC": ajuste.aic, "BIC": ajuste.bic, "convergiu": convergiu,
                  "avisos": " | ".join(sorted({str(a.message) for a in alertas}))})
    scores.append(score if convergiu else np.inf)

melhores_parametros["SARIMAX"] = candidatos_sarimax[int(np.argmin(scores))]
ajuste_sarimax, preparador_sarimax = ajustar_sarimax(idx_treino, melhores_parametros["SARIMAX"])
modelos["SARIMAX"] = ajuste_sarimax
previsoes["SARIMAX"] = prever_sarimax_walk_forward(ajuste_sarimax, preparador_sarimax, idx_treino, idx_teste)
tempos["SARIMAX"] = perf_counter() - inicio
display(pd.DataFrame([r for r in busca if r["modelo"] == "SARIMAX"]).sort_values(["MAE_validação", "BIC"]))

## 6. Holt-Winters


In [ ]:
def ajustar_holt_winters(indices, parametros):
    treino = y.loc[indices].ffill().dropna() / 100
    return ExponentialSmoothing(treino, trend=parametros["trend"],
        damped_trend=parametros["damped_trend"], seasonal=parametros["seasonal"],
        seasonal_periods=SEASONAL_PERIOD, initialization_method="estimated").fit(optimized=True, use_brute=False)

def prever_holt_winters_walk_forward(ajuste, futuro):
    # Prevê t, observa y_t, atualiza nível/tendência/sazonalidade e avança a origem.
    p = ajuste.params
    alpha, gamma = p["smoothing_level"], p["smoothing_seasonal"]
    tem_tendencia = ajuste.model.trend is not None
    beta = p["smoothing_trend"] if tem_tendencia else 0.
    phi = p["damping_trend"] if ajuste.model.damped_trend else 1.
    nivel = float(ajuste.level.iloc[-1])
    tendencia = float(ajuste.trend.iloc[-1]) if tem_tendencia else 0.
    sazonal = np.asarray(ajuste.season.iloc[-SEASONAL_PERIOD:]).copy()
    multiplicativo = ajuste.model.seasonal == "mul"
    saida = []
    for i, real in enumerate(y.loc[futuro] / 100):
        j = i % SEASONAL_PERIOD
        base = nivel + phi * tendencia
        estimativa = base * sazonal[j] if multiplicativo else base + sazonal[j]
        saida.append(estimativa * 100)
        if pd.isna(real):
            nivel, tendencia = base, phi * tendencia
            continue
        novo_nivel = alpha * (real / sazonal[j] if multiplicativo else real - sazonal[j]) + (1-alpha) * base
        nova_tendencia = beta * (novo_nivel-nivel) + (1-beta) * phi*tendencia if tem_tendencia else 0.
        if multiplicativo:
            sazonal[j] = gamma * (real / max(base, 1e-8)) + (1-gamma) * sazonal[j]
        else:
            sazonal[j] = gamma * (real - base) + (1-gamma) * sazonal[j]
        nivel, tendencia = novo_nivel, nova_tendencia
    return np.asarray(saida)

candidatos_hw = [
    {"trend": None, "damped_trend": False, "seasonal": "add"},
    {"trend": "add", "damped_trend": False, "seasonal": "add"},
    {"trend": "add", "damped_trend": True, "seasonal": "add"},
    {"trend": "add", "damped_trend": True, "seasonal": "mul"},
]
inicio = perf_counter()
scores = []
for parametros in candidatos_hw:
    ajuste = ajustar_holt_winters(idx_ajuste, parametros)
    score = mae_observado(idx_validacao, prever_holt_winters_walk_forward(ajuste, idx_validacao))
    busca.append({"modelo": "Holt-Winters", "parâmetros": parametros, "MAE_validação": score,
                  "AIC": ajuste.aic, "BIC": ajuste.bic})
    scores.append(score)
melhores_parametros["Holt-Winters"] = candidatos_hw[int(np.argmin(scores))]
ajuste_hw = ajustar_holt_winters(idx_treino, melhores_parametros["Holt-Winters"])
modelos["Holt-Winters"] = ajuste_hw
previsoes["Holt-Winters"] = prever_holt_winters_walk_forward(ajuste_hw, idx_teste)
tempos["Holt-Winters"] = perf_counter() - inicio
display(pd.DataFrame([r for r in busca if r["modelo"] == "Holt-Winters"]).sort_values("MAE_validação"))

## 7. Random Forest


In [ ]:
modelos_validacao = {}

def ajustar_modelo_ml(nome, candidatos):
        inicio = perf_counter()
        indices_ajuste = idx_ajuste[y.loc[idx_ajuste].notna()]
        if nome == "SVR" and len(indices_ajuste) > 6000:
            indices_ajuste = indices_ajuste[np.linspace(0, len(indices_ajuste) - 1, 6000, dtype=int)]
        scores = []
        candidatos_ajustados = []
        for parametros in candidatos:
            modelo = criar_modelo(nome, parametros)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                modelo.fit(X.loc[indices_ajuste], y.loc[indices_ajuste])
            score = mae_observado(idx_validacao, prever_ml_walk_forward(modelo, idx_validacao))
            busca.append({"modelo": nome, "parâmetros": parametros, "MAE_validação": score})
            scores.append(score)
            candidatos_ajustados.append(modelo)
        melhor = int(np.argmin(scores))
        melhores_parametros[nome] = candidatos[melhor]
        modelos_validacao[nome] = candidatos_ajustados[melhor]
        modelo = criar_modelo(nome, melhores_parametros[nome])
        indices_treino = idx_treino[y.loc[idx_treino].notna()]
        if nome == "SVR" and len(indices_treino) > 8000:
            indices_treino = indices_treino[np.linspace(0, len(indices_treino) - 1, 8000, dtype=int)]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            modelo.fit(X.loc[indices_treino], y.loc[indices_treino])
        modelos[nome] = modelo
        previsoes[nome] = prever_ml_walk_forward(modelo, idx_teste)
        tempos[nome] = perf_counter() - inicio

ajustar_modelo_ml("Random Forest", grades["Random Forest"])


## 8. SVR


In [ ]:
ajustar_modelo_ml("SVR", grades["SVR"])

display(pd.DataFrame([r for r in busca if r["modelo"] in grades]).sort_values(["modelo", "MAE_validação"]))


## 9. MAE, residuos, ACF e Ljung-Box


In [ ]:
assert set(modelos) == {"SARIMAX", "Holt-Winters", "Random Forest", "SVR"}
assert previsoes.drop(columns="valor_real").notna().all().all()
avaliaveis = previsoes["valor_real"].notna()
metricas = pd.DataFrame([
    {"base": BASE_ID, "modelo": nome,
     "MAE": mean_absolute_error(previsoes.loc[avaliaveis, "valor_real"], previsoes.loc[avaliaveis, nome]),
     "horas_avaliadas": int(avaliaveis.sum()), "tempo_segundos": tempos[nome],
     "estrategia": "walk_forward_1_passo_origem_movel",
     "parametros": json.dumps(melhores_parametros[nome], ensure_ascii=False)}
    for nome in modelos
]).sort_values("MAE").reset_index(drop=True)
metricas["ranking"] = np.arange(1, len(metricas)+1)
display(metricas.round(3))

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(metricas.modelo[::-1], metricas.MAE[::-1], color="#167d9a")
ax.set(xlabel="MAE (µg/m³)", title="Base 03 — walk-forward de uma hora")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "ranking_mae.png", dpi=160)
plt.show()

fig, axs = plt.subplots(2, 2, figsize=(15, 8), sharex=True)
recorte = previsoes.iloc[:168]
for ax, nome in zip(axs.flat, modelos):
    ax.plot(recorte.index, recorte.valor_real, color="black", linewidth=1, label="Real")
    ax.plot(recorte.index, recorte[nome], alpha=.8, label="Previsão")
    ax.set_title(nome)
axs[0, 0].legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "previsoes_168h.png", dpi=160)
plt.show()

In [ ]:
residuos = previsoes.drop(columns="valor_real").rsub(previsoes.valor_real, axis=0)
linhas_ljung = []
fig, axs = plt.subplots(4, 3, figsize=(17, 14))
for linha, nome in enumerate(modelos):
    erro = residuos.loc[avaliaveis, nome].dropna()
    axs[linha, 0].plot(erro, linewidth=.5)
    axs[linha, 0].axhline(0, color="black", linewidth=.6)
    axs[linha, 0].set_title(f"{nome} — resíduos")
    plot_acf(erro, lags=48, zero=False, ax=axs[linha, 1], title=f"{nome} — ACF")
    stats.probplot(erro, dist="norm", plot=axs[linha, 2])
    axs[linha, 2].set_title(f"{nome} — Q-Q")
    teste = acorr_ljungbox(erro, lags=[24, 48], return_df=True)
    for lag, valores in teste.iterrows():
        linhas_ljung.append({"base": BASE_ID, "modelo": nome, "lag": lag,
                            "lb_stat": valores.lb_stat, "p_valor": valores.lb_pvalue,
                            "autocorrelacao_5pct": valores.lb_pvalue < .05})
fig.tight_layout()
fig.savefig(FIGURES_DIR / "diagnostico_residuos.png", dpi=160)
plt.show()
ljung_box = pd.DataFrame(linhas_ljung)
display(ljung_box.round(4))

## 10. Importancia e exportacao


In [ ]:
importancias = []
idx_importancia = idx_validacao[y.loc[idx_validacao].notna()][::24]
for nome in ["Random Forest", "SVR"]:
    resultado = permutation_importance(modelos_validacao[nome], X.loc[idx_importancia], y.loc[idx_importancia],
        scoring="neg_mean_absolute_error", n_repeats=5, random_state=RANDOM_STATE, n_jobs=1)
    for feature, media, desvio in zip(X.columns, resultado.importances_mean, resultado.importances_std):
        importancias.append({"base": BASE_ID, "modelo": nome, "feature": feature,
                             "importancia": media, "desvio": desvio, "metodo": "permutation_MAE_validacao"})

parametros_sarimax = pd.Series(np.asarray(ajuste_sarimax.params), index=ajuste_sarimax.param_names)
coeficientes = pd.DataFrame({
    "feature": EXOG_SARIMAX,
    "importancia": [parametros_sarimax.get(f"x{i}", np.nan) for i in range(1, len(EXOG_SARIMAX)+1)],
})
for _, linha in coeficientes.iterrows():
    importancias.append({"base": BASE_ID, "modelo": "SARIMAX", "feature": linha.feature,
                         "importancia": linha.importancia, "desvio": np.nan, "metodo": "coeficiente_padronizado"})
importancia_features = pd.DataFrame(importancias)
display(importancia_features.sort_values(["modelo", "importancia"], ascending=[True, False]).groupby("modelo").head(10))

fig, axs = plt.subplots(1, 2, figsize=(15, 5))
for ax, nome in zip(axs, ["Random Forest", "SVR"]):
    tabela = importancia_features.query("modelo == @nome").nlargest(10, "importancia").sort_values("importancia")
    ax.barh(tabela.feature, tabela.importancia, xerr=tabela.desvio)
    ax.set(title=nome, xlabel="Aumento do MAE na validação")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "feature_importance.png", dpi=160)
plt.show()

In [ ]:
dados_processados = pd.concat([dados, X.add_prefix("feature__")], axis=1)
dados_processados.to_parquet(PROCESSED_DIR / "base_03.parquet")

predicoes_longas = []
for nome in modelos:
    bloco = pd.DataFrame({
        "base": BASE_ID,
        "modelo": nome,
        "data_origem": idx_teste - pd.Timedelta(hours=HORIZON),
        "data_prevista": idx_teste,
        "horizonte": HORIZON,
        "estrategia": "walk_forward_1_passo_origem_movel",
        "valor_real": previsoes.valor_real.to_numpy(),
        "valor_previsto": previsoes[nome].to_numpy(),
    })
    bloco["residuo"] = bloco.valor_real - bloco.valor_previsto
    predicoes_longas.append(bloco)
predicoes_longas = pd.concat(predicoes_longas, ignore_index=True)

residuos_longos = predicoes_longas[["base", "modelo", "data_origem", "data_prevista", "horizonte", "residuo"]]
predicoes_longas.to_csv(PREDICTIONS_DIR / "base_03_predictions.csv", index=False)
metricas.to_csv(METRICS_DIR / "base_03_metrics.csv", index=False)
residuos_longos.to_csv(RESIDUALS_DIR / "base_03_residuals.csv", index=False)
importancia_features.to_csv(IMPORTANCE_DIR / "base_03_feature_importance.csv", index=False)

arquivos_saida = [
    PROCESSED_DIR / "base_03.parquet",
    PREDICTIONS_DIR / "base_03_predictions.csv",
    METRICS_DIR / "base_03_metrics.csv",
    RESIDUALS_DIR / "base_03_residuals.csv",
    IMPORTANCE_DIR / "base_03_feature_importance.csv",
]
assert all(p.exists() and p.stat().st_size > 0 for p in arquivos_saida)
display(pd.Series({p.name: p.stat().st_size for p in arquivos_saida}, name="bytes"))

### Tabela de comparação dos modelos - walk-forward

|   ranking | modelo        |    MAE |   RMSE |   MAPE_% |    R2 |   vies_medio |   horas_avaliadas |   tempo_segundos |
|----------:|:--------------|-------:|-------:|---------:|------:|-------------:|------------------:|-----------------:|
|         1 | Random Forest |  9.926 | 18.051 |   27.088 | 0.954 |        0.093 |              6906 |           91.283 |
|         2 | SARIMAX       |  9.945 | 17.919 |   28.726 | 0.954 |       -0.072 |              6906 |          290.961 |
|         3 | Holt-Winters  | 10.371 | 19.073 |   27.202 | 0.948 |       -0.001 |              6906 |           32.447 |
|         4 | SVR           | 13.716 | 26.528 |   40.933 | 0.9   |        1.266 |              6906 |           17.08  |


In [ ]:
comparacao_modelos = pd.read_csv(METRICS_DIR / 'base_03_comparacao_modelos.csv')
comparacao_modelos
